# 01 — Data Acquisition: ERA5 (C3S) and CAMS

Downloads climate and air quality data for Berlin (2018–2026) via the Copernicus CDS API.

**Before running:**
1. Register at https://cds.climate.copernicus.eu
2. Go to your profile → API key, copy your UID and key
3. Run the setup cell below to create `~/.cdsapirc`

Data will be saved to:
- `../data/raw/era5/era5_berlin_YYYY.nc` — one file per year
- `../data/raw/cams/cams_berlin_YYYY.nc` — one file per year

In [1]:
from pathlib import Path
import os

ERA5_DIR  = Path('../data/raw/era5');  ERA5_DIR.mkdir(parents=True, exist_ok=True)
CAMS_DIR  = Path('../data/raw/cams');  CAMS_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR  = Path('../data/processed'); PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Directories ready')

Directories ready


## 1. CDS API Credentials

Fill in your `UID` and `API_KEY` from https://cds.climate.copernicus.eu/profile

In [ ]:
CDS_UID     = 'YOUR_UID_HERE'
CDS_API_KEY = 'YOUR_API_KEY_HERE'

cdsapirc = Path.home() / '.cdsapirc'
if not cdsapirc.exists():
    cdsapirc.write_text(f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_UID}:{CDS_API_KEY}\n')
    print(f'Created {cdsapirc}')
else:
    print(f'{cdsapirc} already exists — using existing credentials')

In [ ]:
import cdsapi
c = cdsapi.Client()
print('CDS client initialised successfully')

## 2. Download ERA5 — Single Levels (C3S)

Variables: 2m temperature, 2m dewpoint, precipitation, u/v wind, solar radiation.
Resolution: 0.25° × 0.25° (~28 km), hourly → we resample to daily in notebook 03.
Berlin bounding box: N=52.7, W=13.0, S=52.3, E=13.8

In [ ]:
ERA5_VARIABLES = [
    '2m_temperature',
    '2m_dewpoint_temperature',
    'total_precipitation',
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    'surface_solar_radiation_downwards',
]

BERLIN_BBOX = [52.7, 13.0, 52.3, 13.8]  # [N, W, S, E]

YEARS = [str(y) for y in range(2018, 2027)]
MONTHS = [f'{m:02d}' for m in range(1, 13)]
DAYS   = [f'{d:02d}' for d in range(1, 32)]
HOURS  = [f'{h:02d}:00' for h in range(24)]

In [ ]:
# Download one file per year to keep file sizes manageable (~200 MB each)
for year in YEARS:
    out_path = ERA5_DIR / f'era5_berlin_{year}.nc'
    if out_path.exists():
        print(f'{year}: already downloaded, skipping')
        continue
    print(f'Downloading ERA5 {year}...')
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': ERA5_VARIABLES,
            'year': year,
            'month': MONTHS,
            'day': DAYS,
            'time': HOURS,
            'area': BERLIN_BBOX,
            'format': 'netcdf',
        },
        str(out_path)
    )
    print(f'  Saved → {out_path}')

print('ERA5 download complete')

## 3. Download CAMS — European Air Quality Reanalysis

Variables: PM2.5, PM10, NO2, O3, SO2 — daily means at 0.1° resolution.

**Note on dataset name**: CAMS European air quality reanalysis is available on the ADS (Atmosphere Data Store), which uses the same `cdsapi` client but a different URL. If your `.cdsapirc` points to CDS, you may need a second config for ADS (https://ads.atmosphere.copernicus.eu). Check the dataset page for current access instructions.

In [ ]:
import cdsapi
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time



def make_client(store: str):
    if store == "cds":
        return cdsapi.Client(
            url="https://cds.climate.copernicus.eu/api",
            key="44c8e973-6913-4ba0-bffc-a5228c09d7a4",
        )
    elif store == "ewds":
        return cdsapi.Client(
            url="https://ewds.climate.copernicus.eu/api",
            key="44c8e973-6913-4ba0-bffc-a5228c09d7a4",
        )
    elif store == "ads":
        return cdsapi.Client(
            url="https://ads.atmosphere.copernicus.eu/api",
            key="67325afa-ae1d-480f-ba8a-62fc21b664c4"
        )
    else:
        raise ValueError("store must be 'cds', 'ewds', or 'ads'")



# -----------------------------
# Settings
# -----------------------------
OUT_DIR = Path("/fs2/toby/CmP/data/raw/cams_analysis")

calendar_range = pd.date_range(
    start="2025-01-01",
    end="2026-01-01",
    freq="D"
)

MAX_WORKERS = 3   # start safely with 3



# -----------------------------
# Request builder
# -----------------------------
def build_request(date):
    date_str = date.strftime("%Y-%m-%d")

    request = {
        "variable": [
            "ammonia",
            "carbon_monoxide",
            "formaldehyde",
            "glyoxal",
            "nitrogen_dioxide",
            "nitrogen_monoxide",
            "non_methane_vocs",
            "ozone",
            "particulate_matter_2.5um",
            "pm2.5_ammonium",
            "pm2.5_nitrate",
            "residential_elementary_carbon",
            "secondary_inorganic_aerosol",
            "pm2.5_sulphate",
            "total_elementary_carbon",
            "pm2.5_total_organic_matter",
            "particulate_matter_10um",
            "dust",
            "pm10_sea_salt_dry",
            "pm10_wildfires",
            "peroxyacyl_nitrates",
            "sulphur_dioxide"
        ],
        "model": ["ensemble"],
        "level": ["50"],
        "date": [f"{date_str}/{date_str}"],
        "type": ["analysis"],
        "time": [
            "00:00", "01:00", "02:00",
            "03:00", "04:00", "05:00",
            "06:00", "07:00", "08:00",
            "09:00", "10:00", "11:00",
            "12:00", "13:00", "14:00",
            "15:00", "16:00", "17:00",
            "18:00", "19:00", "20:00",
            "21:00", "22:00", "23:00"
        ],
        "leadtime_hour": ["0"],
        "data_format": "netcdf_zip",

        # Berlin box: North, West, South, East
        "area": [52.68, 13.09, 52.34, 13.76]
    }

    return request


# -----------------------------
# Single-date download function
# -----------------------------
def download_one_day(date, retries=3, sleep_seconds=10):
    dataset = "cams-europe-air-quality-forecasts"
    date_str = date.strftime("%Y-%m-%d")

    output_file = Path(OUT_DIR) / f"ads_cams_forecast_{date_str}.zip"
    temp_file = Path(OUT_DIR) / f"ads_cams_forecast_{date_str}.zip.tmp"

    # Skip already downloaded files
    if output_file.exists() and output_file.stat().st_size > 0:
        return date_str, "skipped"

    request = build_request(date)

    for attempt in range(1, retries + 1):
        try:
            client = make_client("ads")

            client.retrieve(dataset, request).download(str(temp_file))

            # Rename only after successful download
            temp_file.rename(output_file)

            return date_str, "downloaded"

        except Exception as e:
            if temp_file.exists():
                temp_file.unlink()

            if attempt < retries:
                time.sleep(sleep_seconds)
            else:
                return date_str, f"failed: {e}"


# -----------------------------
# Parallel download
# -----------------------------
results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_one_day, date): date
        for date in calendar_range
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading CAMS"):
        result = future.result()
        results.append(result)


# -----------------------------
# Summary
# -----------------------------
downloaded = [r for r in results if r[1] == "downloaded"]
skipped = [r for r in results if r[1] == "skipped"]
failed = [r for r in results if r[1].startswith("failed")]

print(f"Downloaded: {len(downloaded)}")
print(f"Skipped: {len(skipped)}")
print(f"Failed: {len(failed)}")

if failed:
    print("\nFailed dates:")
    for date_str, error in failed:
        print(date_str, "->", error)

2026-05-10 06:42:58,803 INFO Request ID is 9558ad2c-649b-4c16-bc0c-43dac1a033e4
2026-05-10 06:42:58,886 INFO Request ID is 38bf15e5-d18a-4f88-93b6-f1f2c70e9acc
2026-05-10 06:42:59,006 INFO status has been updated to accepted
2026-05-10 06:42:59,097 INFO status has been updated to accepted
2026-05-10 06:42:59,216 INFO Request ID is 590bd1e7-1c64-42c3-9695-f384f84af2ad
2026-05-10 06:42:59,433 INFO status has been updated to accepted
2026-05-10 06:43:32,720 INFO status has been updated to running
2026-05-10 06:43:32,777 INFO status has been updated to running
2026-05-10 06:43:34,079 INFO status has been updated to running
2026-05-10 06:44:15,891 INFO status has been updated to successful
2026-05-10 06:44:16,278 INFO status has been updated to successful


7d33d6a2dc1d228283d18a009df6b4e7.zip:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

7fc308a84db6282e4f30f43e6774250d.zip:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

2026-05-10 06:44:17,471 INFO status has been updated to successful


8dca0eeeaffa6598cd30cb64570bea55.zip:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

...

2026-05-10 07:03:40,999 INFO Request ID is 3c3a6615-7498-4937-ba83-e2b75de255f3
2026-05-10 07:03:41,208 INFO status has been updated to accepted
2026-05-10 07:04:00,207 INFO status has been updated to running
2026-05-10 07:04:00,319 INFO status has been updated to running
2026-05-10 07:04:15,467 INFO status has been updated to running
2026-05-10 07:04:17,633 INFO status has been updated to rejected
2026-05-10 07:04:29,989 INFO Request ID is 30a8b49f-8321-4bb0-b6e8-176f3a359fc4
2026-05-10 07:04:30,193 INFO status has been updated to accepted
2026-05-10 07:04:44,593 INFO status has been updated to rejected
2026-05-10 07:04:47,539 INFO status has been updated to successful


3fc370ba298c4a9ffa210289618aab3e.zip:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

2026-05-10 07:04:52,064 INFO Request ID is 4bd7a7db-6128-40cb-be32-7a67abb98900
2026-05-10 07:04:52,260 INFO status has been updated to accepted
2026-05-10 07:04:58,665 INFO status has been updated to successful
2026-05-10 07:04:58,863 INFO Request ID is 2d9110bd-9202-4bc1-8460-d50dd2aa37b8
2026-05-10 07:04:59,068 INFO status has been updated to accepted


d5cefe57c5f75f74eac9d924b91cc1a0.zip:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

2026-05-10 07:05:02,555 INFO Request ID is f2ae25b9-228d-41c6-9784-b0df279de3a2
2026-05-10 07:05:02,771 INFO status has been updated to accepted


## 4. Verification — Check Downloaded Files

In [ ]:
import xarray as xr

era5_files = sorted(ERA5_DIR.glob('era5_berlin_*.nc'))
cams_files = sorted(CAMS_DIR.glob('cams_berlin_*.nc'))

print(f'ERA5 files: {len(era5_files)}')
for f in era5_files:
    ds = xr.open_dataset(f)
    t = ds.time.values
    print(f'  {f.name}: {len(t)} timesteps  {str(t[0])[:10]} → {str(t[-1])[:10]}  vars={list(ds.data_vars)}')
    ds.close()

print(f'\nCAMS files: {len(cams_files)}')
for f in cams_files:
    ds = xr.open_dataset(f)
    t = ds.time.values
    print(f'  {f.name}: {len(t)} timesteps  {str(t[0])[:10]} → {str(t[-1])[:10]}  vars={list(ds.data_vars)}')
    ds.close()